# Notebook 09: Official 3DGS Code Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/09_official_code_walkthrough.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the official 3DGS repository structure
2. Learn how the CUDA rasterizer works
3. Understand the integration with COLMAP for initialization
4. Walk through the training and rendering code
5. Know how to extend and modify the codebase

**Estimated Time**: 90 minutes

**Prerequisites**: All previous notebooks (00-08)

---

## Setup

In [ ]:
import os
import sys

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Path setup
for path in ['../../src', '../src', './src']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')):
        sys.path.insert(0, full_path)
        break

import numpy as np
import torch
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("Setup complete!")

## 1. Official Repository Overview

### Repository: graphdeco-inria/gaussian-splatting

```
gaussian-splatting/
├── arguments/              # Command-line argument definitions
│   ├── __init__.py
│   └── ...                 # ModelParams, OptimizationParams, etc.
├── scene/                  # Scene and Gaussian management
│   ├── __init__.py         # Scene class
│   ├── gaussian_model.py   # GaussianModel class (THE CORE)
│   ├── dataset_readers.py  # COLMAP, Blender readers
│   └── cameras.py          # Camera models
├── utils/                  # Utility functions
│   ├── loss_utils.py       # L1, SSIM losses
│   ├── image_utils.py      # Image processing
│   ├── general_utils.py    # Misc utilities
│   └── graphics_utils.py   # Focal length, fov conversions
├── submodules/
│   ├── diff-gaussian-rasterization/  # CUDA rasterizer (KEY)
│   └── simple-knn/         # KNN for initialization
├── train.py                # Main training script
├── render.py               # Rendering script
├── convert.py              # COLMAP conversion
└── metrics.py              # Evaluation metrics (PSNR, SSIM, LPIPS)
```

### Key Components

| Component | File | Purpose |
|-----------|------|---------|
| GaussianModel | `scene/gaussian_model.py` | Core Gaussian parameters |
| Scene | `scene/__init__.py` | Scene management |
| Rasterizer | `submodules/diff-gaussian-rasterization/` | CUDA rendering |
| Training | `train.py` | Training loop |
| Rendering | `render.py` | Inference rendering |

In [ ]:
# Visualize the repository structure
repo_structure = """
┌─────────────────────────────────────────────────────────────┐
│                    OFFICIAL 3DGS STRUCTURE                   │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│   ┌──────────────┐      ┌──────────────┐                    │
│   │   train.py   │      │  render.py   │                    │
│   │   (entry)    │      │  (inference) │                    │
│   └──────┬───────┘      └──────┬───────┘                    │
│          │                      │                            │
│          ▼                      ▼                            │
│   ┌──────────────────────────────────────┐                  │
│   │             scene/                    │                  │
│   │  ┌────────────────────────────────┐  │                  │
│   │  │       GaussianModel            │  │                  │
│   │  │  - means (xyz)                 │  │                  │
│   │  │  - scales                      │  │                  │
│   │  │  - rotations (quaternions)     │  │                  │
│   │  │  - opacities                   │  │                  │
│   │  │  - features_dc (SH degree 0)   │  │                  │
│   │  │  - features_rest (SH 1-3)      │  │                  │
│   │  └────────────────────────────────┘  │                  │
│   │  ┌────────────────────────────────┐  │                  │
│   │  │          Scene                 │  │                  │
│   │  │  - cameras                     │  │                  │
│   │  │  - point_cloud                 │  │                  │
│   │  │  - gaussians                   │  │                  │
│   │  └────────────────────────────────┘  │                  │
│   └──────────────────────────────────────┘                  │
│                         │                                    │
│                         ▼                                    │
│   ┌──────────────────────────────────────┐                  │
│   │     diff-gaussian-rasterization      │                  │
│   │  ┌────────────────────────────────┐  │                  │
│   │  │      CUDA Rasterizer           │  │                  │
│   │  │  - Tile-based rendering        │  │                  │
│   │  │  - Differentiable splatting    │  │                  │
│   │  │  - Backward pass               │  │                  │
│   │  └────────────────────────────────┘  │                  │
│   └──────────────────────────────────────┘                  │
│                                                              │
└─────────────────────────────────────────────────────────────┘
"""
print(repo_structure)

## 2. GaussianModel Class Analysis

The heart of 3DGS is the `GaussianModel` class in `scene/gaussian_model.py`.

### Key Attributes

```python
class GaussianModel:
    def __init__(self, sh_degree: int):
        self.active_sh_degree = 0
        self.max_sh_degree = sh_degree
        
        # Learnable parameters (nn.Parameter)
        self._xyz = torch.empty(0)           # [N, 3] positions
        self._features_dc = torch.empty(0)   # [N, 1, 3] SH degree 0
        self._features_rest = torch.empty(0) # [N, 15, 3] SH degrees 1-3
        self._scaling = torch.empty(0)       # [N, 3] log-scale
        self._rotation = torch.empty(0)      # [N, 4] quaternions
        self._opacity = torch.empty(0)       # [N, 1] logit-opacity
        
        # For densification
        self.xyz_gradient_accum = torch.empty(0)
        self.denom = torch.empty(0)  # count
        self.max_radii2D = torch.empty(0)
```

In [ ]:
# Simulate the official GaussianModel structure
import torch
import torch.nn as nn

class OfficialGaussianModel:
    """
    Simplified recreation of the official GaussianModel class.
    
    This mirrors the structure in gaussian-splatting/scene/gaussian_model.py
    """
    
    def __init__(self, sh_degree: int = 3):
        self.active_sh_degree = 0
        self.max_sh_degree = sh_degree
        
        # Setup functions (activations)
        def build_covariance_from_scaling_rotation(scaling, scaling_modifier, rotation):
            """Build 3D covariance from scale and rotation."""
            L = self._build_scaling_rotation(scaling_modifier * scaling, rotation)
            actual_covariance = L @ L.transpose(-1, -2)
            return actual_covariance
        
        self.scaling_activation = torch.exp
        self.scaling_inverse_activation = torch.log
        self.opacity_activation = torch.sigmoid
        self.inverse_opacity_activation = lambda x: torch.log(x / (1 - x))
        self.rotation_activation = lambda x: torch.nn.functional.normalize(x, dim=-1)
        self.covariance_activation = build_covariance_from_scaling_rotation
        
        # Learnable parameters (empty initially)
        self._xyz = torch.empty(0)
        self._features_dc = torch.empty(0)
        self._features_rest = torch.empty(0)
        self._scaling = torch.empty(0)
        self._rotation = torch.empty(0)
        self._opacity = torch.empty(0)
        
        # For densification
        self.xyz_gradient_accum = torch.empty(0)
        self.denom = torch.empty(0)
        self.max_radii2D = torch.empty(0)
        
        self.spatial_lr_scale = 1.0
    
    def _build_scaling_rotation(self, s, r):
        """Build lower triangular matrix from scale and rotation."""
        r = torch.nn.functional.normalize(r, dim=-1)
        
        # Quaternion to rotation matrix
        w, x, y, z = r[:, 0], r[:, 1], r[:, 2], r[:, 3]
        
        R = torch.zeros((r.shape[0], 3, 3), device=r.device)
        R[:, 0, 0] = 1 - 2*(y*y + z*z)
        R[:, 0, 1] = 2*(x*y - w*z)
        R[:, 0, 2] = 2*(x*z + w*y)
        R[:, 1, 0] = 2*(x*y + w*z)
        R[:, 1, 1] = 1 - 2*(x*x + z*z)
        R[:, 1, 2] = 2*(y*z - w*x)
        R[:, 2, 0] = 2*(x*z - w*y)
        R[:, 2, 1] = 2*(y*z + w*x)
        R[:, 2, 2] = 1 - 2*(x*x + y*y)
        
        # Scale matrix
        S = torch.diag_embed(s)
        
        return R @ S
    
    @property
    def get_xyz(self):
        return self._xyz
    
    @property
    def get_scaling(self):
        return self.scaling_activation(self._scaling)
    
    @property
    def get_rotation(self):
        return self.rotation_activation(self._rotation)
    
    @property
    def get_opacity(self):
        return self.opacity_activation(self._opacity)
    
    @property
    def get_features(self):
        features_dc = self._features_dc
        features_rest = self._features_rest
        return torch.cat([features_dc, features_rest], dim=1)
    
    def create_from_pcd(self, pcd_points, pcd_colors, spatial_lr_scale=1.0):
        """
        Initialize Gaussians from point cloud.
        
        This is called when loading COLMAP data.
        """
        self.spatial_lr_scale = spatial_lr_scale
        N = pcd_points.shape[0]
        
        # Positions
        self._xyz = nn.Parameter(pcd_points.clone())
        
        # SH features from colors
        # DC term: convert RGB [0,1] to SH
        SH_C0 = 0.28209479177387814
        fused_color = (pcd_colors - 0.5) / SH_C0
        self._features_dc = nn.Parameter(
            fused_color.unsqueeze(1).contiguous()  # [N, 1, 3]
        )
        
        # Rest of SH (initialized to zero)
        features_rest = torch.zeros(N, 15, 3)  # degrees 1-3
        self._features_rest = nn.Parameter(features_rest)
        
        # Scales: from nearest neighbor distances
        # In official code: uses simple-knn to compute distances
        dist = torch.ones(N) * 0.01  # Placeholder
        scales = torch.log(dist.unsqueeze(-1).repeat(1, 3))
        self._scaling = nn.Parameter(scales)
        
        # Rotations: identity quaternions
        rots = torch.zeros(N, 4)
        rots[:, 0] = 1  # w=1, x=y=z=0
        self._rotation = nn.Parameter(rots)
        
        # Opacities: inverse sigmoid of 0.1
        opacities = self.inverse_opacity_activation(0.1 * torch.ones(N, 1))
        self._opacity = nn.Parameter(opacities)
        
        print(f"Created {N} Gaussians from point cloud")
        return self
    
    def __len__(self):
        return self._xyz.shape[0]


# Test
model = OfficialGaussianModel(sh_degree=3)

# Simulate point cloud
torch.manual_seed(42)
points = torch.rand(1000, 3) * 10 - 5
colors = torch.rand(1000, 3)

model.create_from_pcd(points, colors)

print(f"\nModel Statistics:")
print(f"  Number of Gaussians: {len(model)}")
print(f"  XYZ shape: {model._xyz.shape}")
print(f"  Features DC shape: {model._features_dc.shape}")
print(f"  Features Rest shape: {model._features_rest.shape}")
print(f"  Scaling shape: {model._scaling.shape}")
print(f"  Rotation shape: {model._rotation.shape}")
print(f"  Opacity shape: {model._opacity.shape}")

## 3. CUDA Rasterizer Architecture

The `diff-gaussian-rasterization` submodule is critical for performance.

### File Structure

```
diff-gaussian-rasterization/
├── cuda_rasterizer/
│   ├── rasterizer.h          # Main interface
│   ├── rasterizer_impl.cu    # CUDA implementation
│   ├── forward.cu            # Forward pass
│   ├── backward.cu           # Backward pass
│   └── auxiliary.h           # Helper functions
├── rasterize_points.cu       # PyTorch binding
├── ext.cpp                   # Python extension entry
└── setup.py                  # Build configuration
```

### Tile-Based Rendering

The rasterizer uses a tile-based approach for efficiency:

1. **Preprocessing**: Project all Gaussians, compute 2D covariances
2. **Tile Assignment**: Assign each Gaussian to overlapping tiles
3. **Sorting**: Sort Gaussians by depth within each tile
4. **Rendering**: Render each tile independently (parallelizable)

In [ ]:
# Visualize tile-based rendering
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Gaussian coverage
ax = axes[0]
ax.set_xlim(0, 256)
ax.set_ylim(256, 0)  # Flip y

# Draw tiles
tile_size = 16
for i in range(0, 256, tile_size):
    for j in range(0, 256, tile_size):
        rect = patches.Rectangle((i, j), tile_size, tile_size,
                                 linewidth=0.5, edgecolor='gray',
                                 facecolor='none')
        ax.add_patch(rect)

# Draw Gaussians
np.random.seed(42)
for _ in range(20):
    cx, cy = np.random.rand(2) * 256
    rx, ry = np.random.rand(2) * 30 + 10
    angle = np.random.rand() * 360
    ellipse = patches.Ellipse((cx, cy), rx, ry, angle=angle,
                              fill=True, alpha=0.3,
                              facecolor=plt.cm.tab10(np.random.randint(10)))
    ax.add_patch(ellipse)

ax.set_title('Step 1: Gaussian Coverage\n(16x16 tile grid)')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Panel 2: Tile assignment
ax = axes[1]
ax.set_xlim(0, 256)
ax.set_ylim(256, 0)

# Highlight specific tiles
highlighted_tiles = [(3, 5), (4, 5), (3, 6), (4, 6), (5, 5)]
for i, j in highlighted_tiles:
    rect = patches.Rectangle((i*tile_size, j*tile_size), tile_size, tile_size,
                             linewidth=2, edgecolor='red',
                             facecolor='red', alpha=0.3)
    ax.add_patch(rect)

# Draw one Gaussian spanning multiple tiles
ellipse = patches.Ellipse((70, 90), 50, 30, angle=30,
                          fill=True, alpha=0.5,
                          facecolor='blue', edgecolor='blue', linewidth=2)
ax.add_patch(ellipse)

# Draw all tiles lightly
for i in range(0, 256, tile_size):
    for j in range(0, 256, tile_size):
        rect = patches.Rectangle((i, j), tile_size, tile_size,
                                 linewidth=0.5, edgecolor='gray',
                                 facecolor='none')
        ax.add_patch(rect)

ax.set_title('Step 2: Tile Assignment\n(Gaussian → affected tiles)')
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Panel 3: Parallel rendering
ax = axes[2]
ax.set_xlim(0, 256)
ax.set_ylim(256, 0)

# Show threads working on different tiles
cmap = plt.cm.get_cmap('tab20')
for i in range(0, 256, tile_size):
    for j in range(0, 256, tile_size):
        color = cmap((i + j) % 20)
        rect = patches.Rectangle((i, j), tile_size, tile_size,
                                 linewidth=0.5, edgecolor='gray',
                                 facecolor=color, alpha=0.6)
        ax.add_patch(rect)

ax.set_title('Step 3: Parallel Rendering\n(Each tile = CUDA thread block)')
ax.set_xlabel('X')
ax.set_ylabel('Y')

plt.suptitle('Tile-Based CUDA Rasterization', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: Tile-based approach enables massive parallelism")
print("  - Each tile is processed by a CUDA thread block")
print("  - Gaussians are sorted by depth within each tile")
print("  - Front-to-back alpha blending for correct occlusion")

## 4. Forward Pass: Rasterization

### Python Interface

```python
# From diff_gaussian_rasterization/__init__.py
def rasterize_gaussians(
    means3D,            # [N, 3] positions
    means2D,            # [N, 2] projected positions (for grad)
    sh,                 # [N, K, 3] SH coefficients
    colors_precomp,     # [N, 3] precomputed colors (optional)
    opacities,          # [N, 1] opacities
    scales,             # [N, 3] scales
    rotations,          # [N, 4] quaternions
    cov3D_precomp,      # [N, 6] precomputed covariances (optional)
    raster_settings,    # GaussianRasterizationSettings
):
    return _RasterizeGaussians.apply(
        means3D, means2D, sh, colors_precomp, opacities,
        scales, rotations, cov3D_precomp, raster_settings
    )
```

### Rasterization Settings

```python
class GaussianRasterizationSettings:
    def __init__(
        self,
        image_height: int,
        image_width: int,
        tanfovx: float,
        tanfovy: float,
        bg: torch.Tensor,       # Background color
        scale_modifier: float,
        viewmatrix: torch.Tensor,
        projmatrix: torch.Tensor,
        sh_degree: int,
        campos: torch.Tensor,   # Camera position (for SH)
        prefiltered: bool,
        debug: bool,
    ): ...
```

In [ ]:
# Simulate the rasterization settings
from dataclasses import dataclass

@dataclass
class GaussianRasterizationSettings:
    """Settings for Gaussian rasterization."""
    image_height: int
    image_width: int
    tanfovx: float
    tanfovy: float
    bg: torch.Tensor
    scale_modifier: float
    viewmatrix: torch.Tensor
    projmatrix: torch.Tensor
    sh_degree: int
    campos: torch.Tensor
    prefiltered: bool = False
    debug: bool = False


# Create example settings
H, W = 800, 800
fov = 60  # degrees
fov_rad = fov * np.pi / 180

settings = GaussianRasterizationSettings(
    image_height=H,
    image_width=W,
    tanfovx=np.tan(fov_rad / 2),
    tanfovy=np.tan(fov_rad / 2),
    bg=torch.ones(3),  # White background
    scale_modifier=1.0,
    viewmatrix=torch.eye(4),
    projmatrix=torch.eye(4),
    sh_degree=3,
    campos=torch.tensor([0., 0., 5.]),
)

print("Rasterization Settings:")
print("=" * 50)
print(f"  Image size: {settings.image_width} x {settings.image_height}")
print(f"  FOV: {fov}° (tan = {settings.tanfovx:.4f})")
print(f"  Background: {settings.bg.tolist()}")
print(f"  SH degree: {settings.sh_degree}")
print(f"  Camera position: {settings.campos.tolist()}")

## 5. Backward Pass: Gradient Computation

The CUDA rasterizer implements custom backward pass for all parameters.

### Gradients Computed

| Parameter | Shape | Gradient Source |
|-----------|-------|-----------------|
| means3D | [N, 3] | Position affects projected 2D position |
| scales | [N, 3] | Scale affects 2D covariance |
| rotations | [N, 4] | Rotation affects 2D covariance |
| opacities | [N, 1] | Opacity affects alpha blending |
| sh | [N, K, 3] | SH affects rendered color |

### Key Insight: Differentiable Splatting

The forward pass stores intermediate values needed for backprop:
- Transmittance values at each pixel
- Which Gaussians contributed to each pixel
- 2D covariance inverses

In [ ]:
# Visualize gradient flow
gradient_flow = """
┌────────────────────────────────────────────────────────────────┐
│                    GRADIENT FLOW IN 3DGS                        │
├────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Loss = (1-λ)·L1 + λ·D-SSIM                                   │
│     ↓                                                           │
│   ∂Loss/∂Image                                                  │
│     ↓                                                           │
│   ┌─────────────────────────────────────────────────────────┐  │
│   │                  Alpha Blending                          │  │
│   │   C_pixel = Σ αᵢ · Tᵢ · cᵢ + T_final · bg               │  │
│   └─────────────────────────────────────────────────────────┘  │
│     │         │         │                                       │
│     ▼         ▼         ▼                                       │
│   ∂/∂αᵢ    ∂/∂cᵢ     ∂/∂Tᵢ                                    │
│     │         │         │                                       │
│     ▼         │         ▼                                       │
│   ┌─────┐     │     ┌─────┐                                    │
│   │ αᵢ  │     │     │ Tᵢ  │                                    │
│   └──┬──┘     │     └──┬──┘                                    │
│      │        │        │                                        │
│      │        ▼        │                                        │
│      │   ┌─────────┐   │                                        │
│      │   │   SH    │   │                                        │
│      │   │eval(d)  │   │                                        │
│      │   └────┬────┘   │                                        │
│      │        │        │                                        │
│      ▼        ▼        ▼                                        │
│   ┌──────────────────────────────────────────────┐             │
│   │           Gaussian Parameters                 │             │
│   │  ┌───┐ ┌───────┐ ┌────────┐ ┌───────┐ ┌───┐ │             │
│   │  │xyz│ │scales │ │rotation│ │opacity│ │ SH│ │             │
│   │  └───┘ └───────┘ └────────┘ └───────┘ └───┘ │             │
│   └──────────────────────────────────────────────┘             │
│                                                                 │
└────────────────────────────────────────────────────────────────┘
"""
print(gradient_flow)

## 6. Training Loop (train.py)

The main training script follows this structure:

In [ ]:
# Pseudo-code for official training loop
training_loop_code = '''
def training(dataset, opt, pipe, testing_iterations, saving_iterations, ...):
    """
    Main training loop from train.py.
    """
    # Initialize
    gaussians = GaussianModel(dataset.sh_degree)
    scene = Scene(dataset, gaussians)
    gaussians.training_setup(opt)  # Setup optimizer
    
    bg = torch.rand(3) if opt.random_background else torch.ones(3)
    
    # Training loop
    for iteration in range(1, opt.iterations + 1):
        
        # 1. Pick random training camera
        viewpoint_cam = scene.getTrainCameras()[randint(0, len(cameras)-1)]
        
        # 2. Render
        render_pkg = render(viewpoint_cam, gaussians, pipe, bg)
        image = render_pkg["render"]        # [3, H, W]
        viewspace_point_tensor = render_pkg["viewspace_points"]
        visibility_filter = render_pkg["visibility_filter"]
        radii = render_pkg["radii"]
        
        # 3. Loss
        gt_image = viewpoint_cam.original_image
        Ll1 = l1_loss(image, gt_image)
        loss = (1.0 - opt.lambda_dssim) * Ll1 + \
               opt.lambda_dssim * (1.0 - ssim(image, gt_image))
        
        # 4. Backward
        loss.backward()
        
        # 5. Densification (500 <= iter <= 15000, every 100 iters)
        if iteration < opt.densify_until_iter:
            # Accumulate gradients
            gaussians.add_densification_stats(
                viewspace_point_tensor, visibility_filter
            )
            
            if iteration % opt.densification_interval == 0:
                # Clone/Split based on gradient
                gaussians.densify_and_prune(
                    opt.densify_grad_threshold,
                    opt.opacity_threshold,
                    scene.cameras_extent,
                )
            
            # Opacity reset
            if iteration % opt.opacity_reset_interval == 0:
                gaussians.reset_opacity()
        
        # 6. Optimizer step
        gaussians.optimizer.step()
        gaussians.optimizer.zero_grad()
        
        # 7. Update learning rate
        gaussians.update_learning_rate(iteration)
        
        # 8. Save checkpoints
        if iteration in saving_iterations:
            scene.save(iteration)
'''

print(training_loop_code)

## 7. COLMAP Integration

### Data Pipeline

```
Images → COLMAP → cameras.bin, images.bin, points3D.bin → 3DGS
```

### COLMAP Files

| File | Content |
|------|---------|
| `cameras.bin` | Camera intrinsics (focal length, principal point) |
| `images.bin` | Camera extrinsics (pose for each image) |
| `points3D.bin` | Sparse 3D points with colors |

In [ ]:
# Visualize COLMAP workflow
colmap_workflow = """
┌─────────────────────────────────────────────────────────────────────┐
│                      COLMAP → 3DGS PIPELINE                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   ┌──────────────┐                                                  │
│   │   Images     │  (Your captured photos)                          │
│   │   *.jpg/png  │                                                  │
│   └──────┬───────┘                                                  │
│          │                                                           │
│          ▼                                                           │
│   ┌──────────────────────────────────────┐                          │
│   │            COLMAP                     │                          │
│   │                                       │                          │
│   │  1. Feature extraction (SIFT)        │                          │
│   │  2. Feature matching                 │                          │
│   │  3. Sparse reconstruction (SfM)      │                          │
│   │  4. Bundle adjustment                │                          │
│   │                                       │                          │
│   └──────────────┬───────────────────────┘                          │
│                  │                                                   │
│                  ▼                                                   │
│   ┌──────────────────────────────────────┐                          │
│   │        sparse/0/                      │                          │
│   │  ├── cameras.bin   (intrinsics)      │                          │
│   │  ├── images.bin    (extrinsics)      │                          │
│   │  └── points3D.bin  (sparse points)   │                          │
│   └──────────────┬───────────────────────┘                          │
│                  │                                                   │
│                  ▼                                                   │
│   ┌──────────────────────────────────────┐                          │
│   │        convert.py                     │                          │
│   │  (Official 3DGS script)              │                          │
│   │  - Reads COLMAP binary files         │                          │
│   │  - Converts to 3DGS format           │                          │
│   │  - Optional: undistort images        │                          │
│   └──────────────┬───────────────────────┘                          │
│                  │                                                   │
│                  ▼                                                   │
│   ┌──────────────────────────────────────┐                          │
│   │        3DGS Scene                     │                          │
│   │  - Cameras with poses                │                          │
│   │  - Initial Gaussians from points     │                          │
│   │  - Ground truth images               │                          │
│   └──────────────────────────────────────┘                          │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
"""
print(colmap_workflow)

In [ ]:
# Simulate COLMAP data reading
import struct
from collections import namedtuple

# COLMAP data structures
CameraModel = namedtuple('CameraModel', ['model_id', 'model_name', 'num_params'])
Camera = namedtuple('Camera', ['id', 'model', 'width', 'height', 'params'])
Image = namedtuple('Image', ['id', 'qvec', 'tvec', 'camera_id', 'name', 'xys', 'point3D_ids'])
Point3D = namedtuple('Point3D', ['id', 'xyz', 'rgb', 'error', 'image_ids', 'point2D_idxs'])

# Camera models supported
CAMERA_MODELS = {
    0: CameraModel(0, 'SIMPLE_PINHOLE', 3),  # f, cx, cy
    1: CameraModel(1, 'PINHOLE', 4),         # fx, fy, cx, cy
    2: CameraModel(2, 'SIMPLE_RADIAL', 4),   # f, cx, cy, k
    3: CameraModel(3, 'RADIAL', 5),          # f, cx, cy, k1, k2
    4: CameraModel(4, 'OPENCV', 8),          # fx, fy, cx, cy, k1, k2, p1, p2
}

print("COLMAP Camera Models:")
print("=" * 50)
for model_id, model in CAMERA_MODELS.items():
    print(f"  {model.model_id}: {model.model_name} ({model.num_params} params)")

# Example: how 3DGS reads cameras
print("\n" + "=" * 50)
print("Example Camera Reading (from dataset_readers.py):")
print("""
def readColmapCameras(cam_extrinsics, cam_intrinsics, images_folder):
    cam_infos = []
    for idx, key in enumerate(cam_extrinsics):
        extr = cam_extrinsics[key]
        intr = cam_intrinsics[extr.camera_id]
        
        # Extract intrinsics
        if intr.model == "SIMPLE_PINHOLE":
            focal_length_x = intr.params[0]
            FovY = focal2fov(focal_length_x, intr.height)
            FovX = focal2fov(focal_length_x, intr.width)
        elif intr.model == "PINHOLE":
            focal_length_x = intr.params[0]
            focal_length_y = intr.params[1]
            FovY = focal2fov(focal_length_y, intr.height)
            FovX = focal2fov(focal_length_x, intr.width)
        ...
        
        # Extract extrinsics (quaternion + translation)
        R = qvec2rotmat(extr.qvec)
        T = np.array(extr.tvec)
        
        cam_info = CameraInfo(
            uid=idx, R=R, T=T,
            FovY=FovY, FovX=FovX,
            image=image, image_path=image_path, image_name=image_name,
            width=intr.width, height=intr.height
        )
        cam_infos.append(cam_info)
    return cam_infos
""")

## 8. Optimizer Configuration

Different parameters have different learning rates:

In [ ]:
# Official optimizer configuration
optimizer_config = {
    'position': {
        'initial_lr': 0.00016,
        'final_lr': 0.0000016,
        'schedule': 'exponential',
        'max_steps': 30000,
    },
    'feature_dc': {
        'lr': 0.0025,
        'schedule': 'constant',
    },
    'feature_rest': {
        'lr': 0.0025 / 20,  # 0.000125
        'schedule': 'constant',
    },
    'opacity': {
        'lr': 0.05,
        'schedule': 'constant',
    },
    'scaling': {
        'lr': 0.005,
        'schedule': 'constant',
    },
    'rotation': {
        'lr': 0.001,
        'schedule': 'constant',
    },
}

print("Official 3DGS Optimizer Configuration:")
print("=" * 60)
for param, config in optimizer_config.items():
    if 'initial_lr' in config:
        print(f"  {param}:")
        print(f"    Initial LR: {config['initial_lr']}")
        print(f"    Final LR: {config['final_lr']}")
        print(f"    Schedule: {config['schedule']}")
    else:
        print(f"  {param}: LR = {config['lr']}, Schedule = {config['schedule']}")

# Visualize LR schedules
fig, ax = plt.subplots(figsize=(12, 5))

iterations = np.arange(0, 30001, 100)

# Position LR (exponential decay)
initial_lr = 0.00016
final_lr = 0.0000016
position_lrs = [initial_lr * (final_lr / initial_lr) ** (i / 30000) for i in iterations]
ax.semilogy(iterations, position_lrs, label='position', linewidth=2)

# Other LRs (constant)
ax.axhline(y=0.05, color='orange', linestyle='--', label='opacity')
ax.axhline(y=0.005, color='green', linestyle='--', label='scaling')
ax.axhline(y=0.0025, color='red', linestyle='--', label='feature_dc')
ax.axhline(y=0.001, color='purple', linestyle='--', label='rotation')
ax.axhline(y=0.000125, color='brown', linestyle='--', label='feature_rest')

ax.set_xlabel('Iteration')
ax.set_ylabel('Learning Rate (log scale)')
ax.set_title('Official 3DGS Learning Rate Schedules')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Densification Implementation

The official densification code in `gaussian_model.py`:

In [ ]:
# Official densification code (simplified)
densification_code = '''
def densify_and_prune(self, max_grad, min_opacity, extent, max_screen_size):
    """
    Densify and prune Gaussians based on gradients and opacity.
    
    From scene/gaussian_model.py
    """
    grads = self.xyz_gradient_accum / self.denom
    grads[grads.isnan()] = 0.0
    
    # Clone small Gaussians with high gradient
    self.densify_and_clone(grads, max_grad, extent)
    
    # Split large Gaussians with high gradient
    self.densify_and_split(grads, max_grad, extent)
    
    # Prune
    prune_mask = (self.get_opacity < min_opacity).squeeze()
    
    if max_screen_size:
        big_points_vs = self.max_radii2D > max_screen_size
        big_points_ws = self.get_scaling.max(dim=1).values > 0.1 * extent
        prune_mask = torch.logical_or(torch.logical_or(
            prune_mask, big_points_vs), big_points_ws)
    
    self.prune_points(prune_mask)
    
    # Reset stats
    torch.cuda.empty_cache()


def densify_and_clone(self, grads, grad_threshold, scene_extent):
    """
    Clone small Gaussians with high gradient.
    """
    # Select Gaussians to clone
    selected_pts_mask = torch.where(
        torch.norm(grads, dim=-1) >= grad_threshold, True, False
    )
    selected_pts_mask = torch.logical_and(
        selected_pts_mask,
        torch.max(self.get_scaling, dim=1).values <= 
        self.percent_dense * scene_extent  # Small enough
    )
    
    # Clone: duplicate parameters
    new_xyz = self._xyz[selected_pts_mask]
    new_features_dc = self._features_dc[selected_pts_mask]
    new_features_rest = self._features_rest[selected_pts_mask]
    new_opacities = self._opacity[selected_pts_mask]
    new_scaling = self._scaling[selected_pts_mask]
    new_rotation = self._rotation[selected_pts_mask]
    
    self.densification_postfix(
        new_xyz, new_features_dc, new_features_rest,
        new_opacities, new_scaling, new_rotation
    )


def densify_and_split(self, grads, grad_threshold, scene_extent, N=2):
    """
    Split large Gaussians with high gradient.
    """
    n_init_points = self.get_xyz.shape[0]
    padded_grad = torch.zeros(n_init_points)
    padded_grad[:grads.shape[0]] = grads.squeeze()
    
    # Select Gaussians to split
    selected_pts_mask = torch.where(
        padded_grad >= grad_threshold, True, False
    )
    selected_pts_mask = torch.logical_and(
        selected_pts_mask,
        torch.max(self.get_scaling, dim=1).values >
        self.percent_dense * scene_extent  # Large enough
    )
    
    # Sample new positions from old Gaussians
    stds = self.get_scaling[selected_pts_mask].repeat(N, 1)
    means = torch.zeros((stds.size(0), 3))
    samples = torch.normal(mean=means, std=stds)
    
    # Transform samples by rotation
    rots = build_rotation(self._rotation[selected_pts_mask]).repeat(N, 1, 1)
    new_xyz = torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1)
    new_xyz += self.get_xyz[selected_pts_mask].repeat(N, 1)
    
    # Reduce scale
    new_scaling = self.scaling_inverse_activation(
        self.get_scaling[selected_pts_mask].repeat(N, 1) / (0.8 * N)
    )
    
    # Copy other parameters
    new_rotation = self._rotation[selected_pts_mask].repeat(N, 1)
    new_features_dc = self._features_dc[selected_pts_mask].repeat(N, 1, 1)
    new_features_rest = self._features_rest[selected_pts_mask].repeat(N, 1, 1)
    new_opacity = self._opacity[selected_pts_mask].repeat(N, 1)
    
    self.densification_postfix(
        new_xyz, new_features_dc, new_features_rest,
        new_opacity, new_scaling, new_rotation
    )
    
    # Remove original split Gaussians
    prune_filter = torch.cat([
        selected_pts_mask,
        torch.zeros(N * selected_pts_mask.sum(), dtype=bool)
    ])
    self.prune_points(prune_filter)
'''

print(densification_code)

## 10. Rendering and Evaluation

### render.py

The rendering script produces novel view images:

In [ ]:
# Rendering pipeline
render_code = '''
def render(viewpoint_camera, pc, pipe, bg_color, scaling_modifier=1.0, ...):
    """
    Render Gaussians from a specific viewpoint.
    
    Args:
        viewpoint_camera: Camera with pose and intrinsics
        pc: GaussianModel with all parameters
        pipe: Pipeline settings
        bg_color: Background color tensor
    
    Returns:
        Dictionary with:
        - render: Rendered image [3, H, W]
        - viewspace_points: 2D positions (for grad)
        - visibility_filter: Which Gaussians are visible
        - radii: 2D radii of each Gaussian
    """
    # Create rasterization settings
    raster_settings = GaussianRasterizationSettings(
        image_height=int(viewpoint_camera.image_height),
        image_width=int(viewpoint_camera.image_width),
        tanfovx=math.tan(viewpoint_camera.FoVx * 0.5),
        tanfovy=math.tan(viewpoint_camera.FoVy * 0.5),
        bg=bg_color,
        scale_modifier=scaling_modifier,
        viewmatrix=viewpoint_camera.world_view_transform,
        projmatrix=viewpoint_camera.full_proj_transform,
        sh_degree=pc.active_sh_degree,
        campos=viewpoint_camera.camera_center,
        prefiltered=False,
        debug=pipe.debug
    )
    
    rasterizer = GaussianRasterizer(raster_settings=raster_settings)
    
    # Get Gaussian parameters
    means3D = pc.get_xyz
    means2D = screenspace_points  # For gradient accumulation
    opacity = pc.get_opacity
    scales = pc.get_scaling
    rotations = pc.get_rotation
    shs = pc.get_features
    
    # Rasterize!
    rendered_image, radii = rasterizer(
        means3D=means3D,
        means2D=means2D,
        shs=shs,
        colors_precomp=None,
        opacities=opacity,
        scales=scales,
        rotations=rotations,
        cov3D_precomp=None,
    )
    
    return {
        "render": rendered_image,
        "viewspace_points": screenspace_points,
        "visibility_filter": radii > 0,
        "radii": radii,
    }
'''

print(render_code)

## 11. Key Files Summary

| File | Key Functions | Purpose |
|------|---------------|---------|
| `train.py` | `training()` | Main training loop |
| `render.py` | `render()`, `render_set()` | Inference rendering |
| `scene/__init__.py` | `Scene` class | Scene management |
| `scene/gaussian_model.py` | `GaussianModel` | Core Gaussian parameters |
| `scene/dataset_readers.py` | `readColmapSceneInfo()` | Data loading |
| `scene/cameras.py` | `Camera` class | Camera models |
| `utils/loss_utils.py` | `l1_loss()`, `ssim()` | Loss functions |
| `utils/graphics_utils.py` | `getWorld2View2()`, `focal2fov()` | Graphics utilities |
| `arguments/__init__.py` | `ModelParams`, `OptimizationParams` | Arguments |

In [ ]:
# Summary visualization
summary = """
┌────────────────────────────────────────────────────────────────────┐
│                    OFFICIAL 3DGS KEY POINTS                         │
├────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  1. GAUSSIAN REPRESENTATION                                        │
│     - Position (xyz): 3 floats                                     │
│     - Covariance: Scale (3) + Rotation quaternion (4)              │
│     - Appearance: SH coefficients (16×3 = 48 for degree 3)         │
│     - Opacity: 1 float (stored as logit)                           │
│     Total: 59 floats per Gaussian                                  │
│                                                                     │
│  2. CUDA RASTERIZER                                                │
│     - Tile-based (16×16 pixels per tile)                           │
│     - Per-tile sorting by depth                                    │
│     - Front-to-back alpha blending                                 │
│     - Fully differentiable                                         │
│                                                                     │
│  3. TRAINING                                                       │
│     - 30,000 iterations default                                    │
│     - Adam optimizer with per-param LR                             │
│     - Loss: (1-λ)·L1 + λ·D-SSIM, λ=0.2                            │
│     - Densification: iterations 500-15000, every 100               │
│     - Opacity reset: every 3000 iterations                         │
│                                                                     │
│  4. DATA REQUIREMENTS                                              │
│     - Images with COLMAP poses                                     │
│     - cameras.bin, images.bin, points3D.bin                        │
│     - Or: Blender synthetic format                                 │
│                                                                     │
│  5. OUTPUT                                                         │
│     - point_cloud.ply: Gaussian parameters                         │
│     - Novel view rendering at real-time rates                      │
│                                                                     │
└────────────────────────────────────────────────────────────────────┘
"""
print(summary)

## 12. Summary

### Official Repository Structure

- `train.py`: Entry point for training
- `render.py`: Inference rendering
- `scene/gaussian_model.py`: Core GaussianModel class
- `submodules/diff-gaussian-rasterization/`: CUDA rasterizer

### Key Implementation Details

1. **Parameters are stored in raw form** (log-scale, logit-opacity, unnormalized quaternions)
2. **Activation functions** convert to usable values
3. **CUDA rasterizer** uses tile-based approach for speed
4. **Densification** uses gradient statistics
5. **COLMAP integration** provides initialization

---

## Key Takeaways

1. The official code is well-organized with clear separation of concerns
2. CUDA rasterizer is the performance-critical component
3. Training uses sophisticated parameter-specific optimization
4. COLMAP provides crucial initialization from images

---

## Next Steps

In the next notebook, we'll train 3DGS on custom data:

**[10_custom_data_training.ipynb](./10_custom_data_training.ipynb)** - Training on Your Own Data